# Variable/Constants & Imports

In [ ]:
import numpy as np
from iminuit import Minuit
from iminuit.cost import ExtendedBinnedNLL
from scipy.stats import expon, norm, uniform, chi2
from collections.abc import Callable
import plotly.graph_objects as go
import inspect

n_bins = 300
N = 31851
TAU_MU = 2.1969811e-6
exp_args = {"a": 1, "A": 0, "tau": TAU_MU}

# Functions

In [26]:

def exp( x , N ,a, A, tau):
    return a*N*expon.cdf(x , A , tau) 



def exp_unif(x, a, b, tau, A):
    return a * N * (expon.cdf(x, A, tau) + b * N * uniform.cdf(x, 0, 8e-6))


def exp_fit(cdf, a, A, tau, count , edges):
    
    cost = ExtendedBinnedNLL(count, edges, cdf)
    n = Minuit(cost, a, A, tau )
    n.fixed["A"] = True
    n.migrad(ncall = 1000000)
    n.hesse()
    return n

def exp_unif_fit(cdf, a, b, tau, A, count , edges):
    
    cost = ExtendedBinnedNLL(count, edges, cdf)
    n = Minuit(cost, a, b, tau, A )
    # n.fixed['N'] = True
    n.fixed['A'] = True
    # n.limits['b'] = (0, 1)
    n.migrad(ncall = 1000000)
    n.hesse()
    return n

def gauss_fit( cdf , N , mu , sigma , count , edges):
    cost = ExtendedBinnedNLL(count, edges, cdf)
    n = Minuit(cost, N = N , mu = mu , sigma = sigma )
    n.migrad(ncall = 1000000)
    n.hesse()
    return n

def function_generator_with_variable_N(func: Callable, N: int):
    sig = inspect.signature(func)
    
    if "x" not in sig.parameters or "N" not in sig.parameters:
        raise SyntaxError("function defined wrongly: it needs an x as first parameter and N as other parameter")
    def wrapper(x, *args, **kwargs):
        return func(x, N, *args, **kwargs)

    wrapper.__signature__ = inspect.Signature(
        [inspect.Parameter('x', inspect.Parameter.POSITIONAL_OR_KEYWORD)] +
        list(inspect.signature(func).parameters.values())[2:]
    )

    return wrapper

def dataset_analysis(dataset: np.ndarray , creator: Callable, bins: int , args: dict) -> Minuit:
    model_function = function_generator_with_variable_N( creator , len(dataset))
    count, edges = np.histogram( dataset , bins=bins) 
    cost = ExtendedBinnedNLL(count, edges, model_function)

    minuit_element =  Minuit(cost, *args.values())
    
    if "A" in args.keys():
        minuit_element.fixed["A"] = True

    return minuit_element

def end(m:Minuit) -> None:
    m.migrad()
    m.hesse()
    display(m)







    

# Dataset

In [27]:
data_0 = np.genfromtxt("Data/timestamp/21_01_2024_17_42.csv", delimiter=',')
data_1 = np.genfromtxt("Data/timestamp/23_01_2026_17_31.csv", delimiter=',')
data_2 = np.genfromtxt("Data/timestamp/29_01_2026_17_20.csv", delimiter=',')
data_3 = np.genfromtxt("Data/timestamp/02_02_2026_17_14.csv", delimiter=',')
data_4 = np.genfromtxt("Data/timestamp/05_02_2026_18_17.csv", delimiter=',')
data = np.concatenate((data_0, data_1, data_2, data_3, data_4 ))


# Analysis

## Hists

In [28]:

# data_2 = data_2[(data_2 > 0.6e-6) & (data_2 < 1.8e-6)]
count, edges = np.histogram(data, bins=n_bins , density=False)
count_1, edges_1 = np.histogram(data_1, bins=n_bins , density=False)
count_2, edges_2 = np.histogram(data_2, bins=n_bins , density=False)
count_3, edges_3 = np.histogram(data_3, bins=n_bins , density=False)
count_4, edges_4 = np.histogram(data_4, bins=n_bins , density=False)
print (len(data), len(data_1))
fig = go.Figure()

fig.add_trace(go.Bar(x=edges[:-1],  y=count   / len(data),   name='total',        width=np.diff(edges)))
fig.add_trace(go.Bar(x=edges_1[:-1], y=count_1 / len(data_1), name='23/01/2026',   width=np.diff(edges_1)))
fig.add_trace(go.Bar(x=edges_2[:-1], y=count_2 / len(data_2), name='29/01/2026',   width=np.diff(edges_2)))
fig.add_trace(go.Bar(x=edges_3[:-1], y=count_3 / len(data_3), name='02/02/2026',   width=np.diff(edges_3)))
fig.add_trace(go.Bar(x=edges_4[:-1], y=count_4 / len(data_4), name='05/02/2026',   width=np.diff(edges_4)))
fig.update_layout(
    xaxis_title='Time (s)',
    yaxis_title='Counts',
    bargap=0
)

fig.show()

31851 9400


In [29]:
k = dataset_analysis( data_1 , exp , 69 , exp_args)
end(k)

┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 216.6 (χ²/ndof = 3.2)      │              Nfcn = 38               │
│ EDM = 1.03e-05 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ a    │   1.027   │   0.011   │            │            │         │         │       │
│ 1 │ A    │    0.0    │    0.1    │            │            │         │         │  yes  │
│ 2 │ tau  │ 2.230e-6  │ 0.029e-6  │            │            │         │         │       │
└───┴──────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌─────┬─────────────────────────────────────┐
│     │           a           A         tau │
├─────┼─────────────────────────────────────┤
│   a │    0.000114           0 37.2538e-12 │
│   A │           0           0           0 │
│ tau │ 37.2538e-12           0    8.32e-16 │
└─────┴─────────────────────────────────────┘

## Full interpolation 

In [ ]:
n_bins = 100
exp_args = {"a": 1, "A": 0, "tau": TAU_MU}

data_refact = data[(data > 7e-8)&(data < 7.1e-6)]## forse si può anche non tagliare dal basso


n = dataset_analysis( data_refact , exp , bins = n_bins , args=exp_args)

end(n)





┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 189.9 (χ²/ndof = 1.9)      │              Nfcn = 47               │
│ EDM = 7.87e-07 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ a    │   1.083   │   0.006   │            │            │         │         │       │
│ 1 │ A    │    0.0    │    0.1    │            │            │         │         │  yes  │
│ 2 │ tau  │ 2.302e-6  │ 0.018e-6  │            │            │         │         │       │
└───┴──────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌─────┬────────────────────────────────────────┐
│     │            a            A          tau │
├─────┼────────────────────────────────────────┤
│   a │     3.92e-05            0 18.96652e-12 │
│   A │            0            0            0 │
│ tau │ 18.96652e-12            0     3.35e-16 │
└─────┴────────────────────────────────────────┘

In [31]:
print(len(data_2),'\n', 'rate di decadimenti muonici tra il 29genn-2febbr:', 1/(len(data_2)/((24*4)*3600)), 'Hz')
print (len(data_3),'\n', 'rate di decadimenti muonici tra il 2-5 febbraio:', 1/(len(data_3)/((24*3+1)*3600)), 'Hz')
print (len(data_4),'\n', 'rate di decadimenti muonici tra il 5-11 febbraio:', 1/(len(data_4)/((24*4+20)*3600)), 'Hz')

6139 
 rate di decadimenti muonici tra il 29genn-2febbr: 56.29581365043167 Hz
4577 
 rate di decadimenti muonici tra il 2-5 febbraio: 57.4175223945816 Hz
8759 
 rate di decadimenti muonici tra il 5-11 febbraio: 47.67667541956844 Hz


## Plot result + Discrepancy analysis

In [40]:
count,edges = np.histogram(data_refact , n_bins)


fig = go.Figure()
fig.add_trace(go.Bar(x=edges[:-1], y=count/len(data_refact), name='total', width=np.diff(edges)))

x_fit = np.linspace(min(data_refact) , max(data_refact) , n_bins)
y_fit = n.values["a"]*expon.pdf( x_fit , loc = n.values["A"] , scale = n.values["tau"])*np.diff(edges)[0]
fig.add_trace(go.Scatter(x=x_fit, y=y_fit, mode='lines', name='fit', line=dict(shape='linear')))

In [49]:
res = count/len(data_refact) - y_fit

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_fit, y=res, mode='markers', error_y=dict(array=np.sqrt(y_fit)), name='residuals'))
print(np.sqrt(y_fit))
fig.show()

[0.17893351 0.17619823 0.17350475 0.17085245 0.1682407  0.16566887
 0.16313635 0.16064255 0.15818687 0.15576873 0.15338756 0.15104278
 0.14873385 0.14646021 0.14422133 0.14201668 0.13984573 0.13770796
 0.13560287 0.13352996 0.13148874 0.12947873 0.12749944 0.1255504
 0.12363116 0.12174126 0.11988025 0.11804769 0.11624314 0.11446618
 0.11271638 0.11099332 0.10929661 0.10762584 0.1059806  0.10436052
 0.1027652  0.10119427 0.09964735 0.09812408 0.0966241  0.09514704
 0.09369257 0.09226033 0.09084998 0.08946119 0.08809363 0.08674698
 0.08542091 0.08411512 0.08282928 0.0815631  0.08031628 0.07908852
 0.07787952 0.07668901 0.07551669 0.0743623  0.07322555 0.07210618
 0.07100392 0.06991851 0.0688497  0.06779722 0.06676083 0.06574028
 0.06473534 0.06374575 0.0627713  0.06181174 0.06086684 0.0599364
 0.05902017 0.05811795 0.05722953 0.05635468 0.05549321 0.05464491
 0.05380957 0.05298701 0.05217702 0.05137941 0.05059399 0.04982058
 0.04905899 0.04830905 0.04757056 0.04684337 0.04612729 0.045422